# Forschungsfrage 3 - Formatierungsfehler: Traditioneller Ansatz (Regex/Parser)

Rein regelbasierte Bereinigung ohne jegliches gelerntes Modell: Für `price` und
`saving` wird die Formatklasse des Rohwerts durch eine **handgeschriebene** Kette
von regulären Ausdrücken/Bedingungen bestimmt (statt - wie im nächsten Notebook -
durch einen trainierten XGBoost-Klassifikator), anschließend wird derselbe
Extraktions-/Berechnungsschritt angewendet. Für die Datumsspalten ist ohnehin nur
ein regelbasierter Parser sinnvoll (praktisch ein einziges Rohformat je Spalte,
vgl. `01_Benchmark_und_GroundTruth_Erstellung.ipynb`).

Dieser Vergleich isoliert exakt **eine** Variable: Bringt ein aus Trainingsdaten
**gelernter** Formatklassifikator einen Vorteil gegenüber **handkodierten**
Entscheidungsregeln, wenn die eigentliche Extraktionslogik (Prozent-Umrechnung,
Spannen-Mittelwert, Ratio-Formel) in beiden Fällen identisch ist?

Bewertung ausschließlich auf dem `test`-Split - identisch zu den anderen beiden
TF3-Notebooks.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import time
import os

os.makedirs("results", exist_ok=True)

df_raw = pd.read_csv("data/rfd_main.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_raw["row_id"] = df_raw.index

def extract_first_number(s):
    m = re.search(r"(\d+(?:\.\d+)?)", str(s))
    return float(m.group(1)) if m else np.nan


## 2. `price`: handkodierte Regex-Kaskade (kein gelerntes Modell)

Dieselben Formatklassen wie im XGBoost-Notebook, hier jedoch durch feste
`if`/`elif`-Bedingungen statt durch einen trainierten Klassifikator erkannt:

In [2]:
def parse_price_regelbasiert(raw_value):
    s = str(raw_value).strip()
    if "%" in s:
        # "prozent"-Fall (z.B. "50%off" als Preisfeld) - nicht als Preis interpretierbar
        return np.nan
    # "preisspanne": ausschließlich zwei durch einen Bindestrich getrennte Zahlen
    if re.fullmatch(r"\s*\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*", s):
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
        return float(np.mean(nums))
    if "free" in s.lower():
        return 0.0  # "wortwert": Free -> 0
    if not re.search(r"\d", s):
        return np.nan  # "wortwert": z.B. "varies" - nicht interpretierbar
    # numerisch, dollar_prefix, waehrungssuffix, mengenangabe, sonstiges -> erste Zahl (best effort)
    return extract_first_number(s)

price_bench = pd.read_csv("benchmark/tf3_format_price.csv")
price_test = price_bench[price_bench["split"] == "test"].reset_index(drop=True).copy()

t0 = time.time()
price_test["price_pred"] = price_test["raw_value"].apply(parse_price_regelbasiert)
price_parse_time = time.time() - t0

exact_match = (price_test["price_pred"].sub(price_test["true_clean_value"]).abs() < 0.01).mean()
valid_rate = price_test["price_pred"].notna().mean()
print(f"price - Exact-Match-Rate: {exact_match:.3f}  Valid-Format-Rate: {valid_rate:.3f}  (n_test={len(price_test)}, {price_parse_time:.3f}s)")

price_test.to_csv("results/tf3_regelbasiert_price_predictions.csv", index=False)


price - Exact-Match-Rate: 0.992  Valid-Format-Rate: 0.996  (n_test=257, 0.002s)


## 3. `saving`: handkodierte Regex-Kaskade + dieselbe Ratio-Formel

Formatunterscheidung reduziert sich hier auf eine einzige Bedingung
(enthält `%` oder nicht) - die Ratio-Berechnung selbst ist identisch zum
XGBoost-Notebook.

In [3]:
raw_price_lookup = df_raw.set_index("row_id")["price"]

def own_price_pipeline_regelbasiert(raw_price_values):
    return raw_price_values.apply(parse_price_regelbasiert)

def parse_saving_ratio_regelbasiert(raw_value, cleaned_price):
    s = str(raw_value)
    if "%" in s:
        m = re.search(r"(\d+(?:\.\d+)?)\s*%", s)
        return float(m.group(1)) / 100.0 if m else np.nan
    amount = extract_first_number(s)
    if pd.isna(amount) or pd.isna(cleaned_price):
        return np.nan
    denom = cleaned_price + amount
    return amount / denom if denom > 0 else np.nan

saving_bench = pd.read_csv("benchmark/tf3_format_saving.csv")
saving_test = saving_bench[saving_bench["split"] == "test"].reset_index(drop=True).copy()

raw_prices_for_saving = raw_price_lookup.loc[saving_test["row_id_raw"]].reset_index(drop=True)

t0 = time.time()
saving_test["own_cleaned_price"] = own_price_pipeline_regelbasiert(raw_prices_for_saving).values
saving_test["saving_pred"] = [
    parse_saving_ratio_regelbasiert(r, p) for r, p in zip(saving_test["raw_value"], saving_test["own_cleaned_price"])
]
saving_parse_time = time.time() - t0

exact_match_saving = (saving_test["saving_pred"].sub(saving_test["true_clean_value"]).abs() < 0.01).mean()
valid_rate_saving = saving_test["saving_pred"].notna().mean()
print(f"saving - Exact-Match-Rate: {exact_match_saving:.3f}  Valid-Format-Rate: {valid_rate_saving:.3f}  (n_test={len(saving_test)}, {saving_parse_time:.3f}s)")

saving_test.to_csv("results/tf3_regelbasiert_saving_predictions.csv", index=False)


saving - Exact-Match-Rate: 0.980  Valid-Format-Rate: 0.980  (n_test=149, 0.002s)


## 4. Datumsspalten: regelbasierter Parser

**Formatierungsfehler:** `expiry` verwendet - anders als `creation_date`/`last_reply` -
den vollen Monatsnamen (`"July 29, 2020"` statt `"Jul 29, 2020"`) und enthält keine
Uhrzeit. Der Parser deckt beide Monatsnamen-Varianten ab.

In [4]:
def parse_raw_date(s):
    if pd.isna(s):
        return pd.NaT
    s2 = re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(s))
    for fmt in ("%b %d, %Y %I:%M %p", "%b %d, %Y", "%B %d, %Y %I:%M %p", "%B %d, %Y"):
        try:
            return pd.to_datetime(s2, format=fmt)
        except ValueError:
            continue
    return pd.NaT  # regelbasierter Parser kennt dieses Muster nicht -> bewusst kein Generic-Fallback

date_bench = pd.read_csv("benchmark/tf3_format_date.csv")
date_test = date_bench[date_bench["split"] == "test"].reset_index(drop=True)

t0 = time.time()
date_test = date_test.copy()
date_test["parsed"] = date_test["raw_value"].apply(parse_raw_date)
date_parse_time = time.time() - t0
date_test["true_parsed"] = pd.to_datetime(date_test["true_clean_value"], errors="coerce")

def dates_match(row):
    if pd.isna(row["parsed"]) or pd.isna(row["true_parsed"]):
        return False
    if row["source_column"] == "expiry":
        return row["parsed"].date() == row["true_parsed"].date()
    return row["parsed"].floor("min") == row["true_parsed"].floor("min")

date_test["match"] = date_test.apply(dates_match, axis=1)
valid_rate_date = date_test["parsed"].notna().mean()
exact_match_date = date_test["match"].mean()
print(f"Datumsspalten gesamt - Exact-Match-Rate: {exact_match_date:.3f}  Valid-Format-Rate: {valid_rate_date:.3f}  (n_test={len(date_test)})")
print(date_test.groupby("source_column")["match"].mean())

date_test.drop(columns=["parsed", "true_parsed"]).to_csv("results/tf3_regelbasiert_date_predictions.csv", index=False)


Datumsspalten gesamt - Exact-Match-Rate: 1.000  Valid-Format-Rate: 1.000  (n_test=885)
source_column
creation_date    1.0
expiry           1.0
last_reply       1.0
Name: match, dtype: float64


## 5. Metriken und Laufzeit-Log speichern

In [5]:
metrics = {
    "experiment": "TF3_Formatierung", "method": "Regelbasiert",
    "price_exact_match": exact_match, "price_valid_rate": valid_rate,
    "saving_exact_match": exact_match_saving, "saving_valid_rate": valid_rate_saving,
    "date_exact_match": exact_match_date, "date_valid_rate": valid_rate_date,
}
with open("results/tf3_regelbasiert_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([
    {"experiment": "TF3_Formatierung_price", "method": "Regelbasiert", "n_items": len(price_test),
     "wall_time_sec": price_parse_time, "input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0,
     "model_name": "handkodierte_regex (kein Modell, kein LLM/API)"},
    {"experiment": "TF3_Formatierung_saving", "method": "Regelbasiert", "n_items": len(saving_test),
     "wall_time_sec": saving_parse_time, "input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0,
     "model_name": "handkodierte_regex (kein Modell, kein LLM/API)"},
    {"experiment": "TF3_Formatierung_date", "method": "Regelbasiert", "n_items": len(date_test),
     "wall_time_sec": date_parse_time, "input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0,
     "model_name": "handkodierte_regex (kein Modell, kein LLM/API)"},
])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf3_regelbasiert_{price,saving,date}_predictions.csv, results/tf3_regelbasiert_metrics.json")


Gespeichert: results/tf3_regelbasiert_{price,saving,date}_predictions.csv, results/tf3_regelbasiert_metrics.json
